# 20. The SLAP for Out-of-Gauge (OOG) Containers Problem

## Tier 2: The Classic Heuristic (Priority-Based OOG Assignment)

### Key assumptions
- Containers are processed based on a composite priority score
- Priority scores consider weight utilization, lashing complexity, stability impact, and crane accessibility
- Dynamic priority updates reflect changing vessel conditions as assignments are made
- Feasibility constraints are enforced during the assignment process
- Real-time decision making is required for operational efficiency

### Approach (step-by-step)
1. **Priority Calculation**: Compute composite priority scores for all container-slot pairs
2. **Dynamic Queue**: Maintain a priority queue of feasible assignments
3. **Greedy Selection**: Select highest priority assignment iteratively
4. **Constraint Updates**: Update remaining capacities and constraints after each assignment
5. **Performance Analysis**: Compare heuristic solution with optimal mathematical solution

### What to look for in the results
- Priority-based assignment sequence with clear justification
- Dynamic priority updates as assignments are made
- Comparison with mathematical optimization results
- Real-time performance characteristics
- Trade-offs between solution quality and computational speed

### Concrete example (from the source)
We'll implement the Priority-Based OOG Assignment Algorithm with:
- Weighted priority function combining multiple factors
- 3 OOG containers with different characteristics
- 4 vessel slots with varying constraints
- Real-time priority updates and dynamic assignment process

In [ ]:
# Import required libraries for heuristic implementation
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import heapq
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional
import warnings
warnings.filterwarnings('ignore')

# Set up professional visualization style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ Libraries imported successfully for priority-based OOG assignment")

In [ ]:
@dataclass
class OOGContainer:
    """Represents an Out-of-Gauge container with specific characteristics"""
    id: int
    weight: float  # tons
    height: float  # meters
    length: float  # meters
    container_type: str  # flat rack, open-top, platform
    lashing_coeff: float  # lashing requirement coefficient
    time_sensitivity: float  # 0-1, higher = more time-sensitive
    handling_difficulty: float  # 0-1, higher = more difficult to handle

@dataclass
class VesselSlot:
    """Represents an available vessel slot with constraints"""
    id: int
    weight_capacity: float  # tons
    height_limit: float  # meters
    bay_id: int  # bay identifier for stability calculations
    crane_distance: float  # distance to nearest crane
    slot_position: str  # forward, aft, midship

@dataclass(order=True)
class PriorityAssignment:
    """Represents a potential assignment with priority score"""
    priority: float
    container_id: int = field(compare=False)
    slot_id: int = field(compare=False)
    score_components: Dict[str, float] = field(default_factory=dict, compare=False)

@dataclass
class HeuristicProblem:
    """Contains the complete OOG container assignment problem for heuristic solving"""
    containers: List[OOGContainer]
    slots: List[VesselSlot]
    cost_matrix: np.ndarray  # assignment costs
    stability_matrix: np.ndarray  # stability factors
    bay_weight_limits: Dict[int, float]  # maximum weight per bay
    priority_weights: Dict[str, float]  # weights for priority calculation

print("✅ Enhanced data structures defined for priority-based heuristic")

In [ ]:
# Define the enhanced problem instance with additional characteristics
containers = [
    OOGContainer(1, 45, 2.8, 12.0, "flat rack", 2.1, 0.7, 0.8),
    OOGContainer(2, 32, 4.2, 6.0, "open-top", 3.8, 0.9, 0.6),
    OOGContainer(3, 38, 2.6, 12.0, "platform", 1.9, 0.5, 0.4)
]

slots = [
    VesselSlot(1, 50, 3.0, 2, 15.0, "forward"),
    VesselSlot(2, 40, 5.0, 3, 12.0, "midship"),
    VesselSlot(3, 45, 3.0, 3, 18.0, "midship"),
    VesselSlot(4, 60, 4.0, 3, 10.0, "aft")
]

# Assignment cost matrix
cost_matrix = np.array([
    [100, 150, 120, 90],   # Container 1 costs
    [80, 70, 110, 85],     # Container 2 costs
    [95, 140, 100, 80]     # Container 3 costs
])

# Stability factors (higher is better)
stability_matrix = np.array([
    [0.8, 0.7, 0.9, 0.85],
    [0.75, 0.8, 0.7, 0.9],
    [0.85, 0.65, 0.8, 0.75]
])

# Bay weight limits
bay_weight_limits = {2: 80, 3: 120}

# Priority weights (from the source formula)
priority_weights = {
    'capacity_utilization': 0.30,  # w1
    'cost_efficiency': 0.25,        # w2
    'stability_contribution': 0.25, # w3
    'lashing_simplicity': 0.20      # w4
}

# Create the complete problem instance
problem = HeuristicProblem(containers, slots, cost_matrix, stability_matrix, 
                          bay_weight_limits, priority_weights)

print(f"📊 Enhanced problem defined: {len(containers)} containers, {len(slots)} slots")
print(f"⚖️  Priority weights: {priority_weights}")
print(f"📋 Container characteristics:")
for container in containers:
    print(f"   C{container.id}: {container.container_type}, {container.weight}t, {container.height}m, "
          f"lashing: {container.lashing_coeff}, sensitivity: {container.time_sensitivity}, difficulty: {container.handling_difficulty}")

In [ ]:
def calculate_priority_score(problem: HeuristicProblem, container_id: int, slot_id: int,
                              current_assignments: Dict[int, int]) -> PriorityAssignment:
    """
    Calculate priority score for a container-slot pair
    
    Args:
        problem: Complete problem instance
        container_id: Container ID
        slot_id: Slot ID
        current_assignments: Current assignments (container_id -> slot_id)
        
    Returns:
        PriorityAssignment with score and components
    """
    container = problem.containers[container_id]
    slot = problem.slots[slot_id]
    
    # Calculate individual score components
    
    # 1. Capacity utilization (higher is better)
    capacity_utilization = container.weight / slot.weight_capacity
    capacity_score = min(capacity_utilization, 1.0)  # Cap at 1.0
    
    # 2. Cost efficiency (lower cost = higher score)
    max_cost = problem.cost_matrix.max()
    cost_score = 1 - (problem.cost_matrix[container_id, slot_id] / max_cost)
    
    # 3. Stability contribution (higher is better)
    stability_score = problem.stability_matrix[container_id, slot_id]
    
    # 4. Lashing simplicity (lower coefficient = higher score)
    max_lashing = max(c.lashing_coeff for c in problem.containers)
    lashing_score = 1 - (container.lashing_coeff / max_lashing)
    
    # Calculate weighted priority score
    priority = (
        problem.priority_weights['capacity_utilization'] * capacity_score +
        problem.priority_weights['cost_efficiency'] * cost_score +
        problem.priority_weights['stability_contribution'] * stability_score +
        problem.priority_weights['lashing_simplicity'] * lashing_score
    )
    
    # Store components for analysis
    score_components = {
        'capacity_utilization': capacity_score,
        'cost_efficiency': cost_score,
        'stability_contribution': stability_score,
        'lashing_simplicity': lashing_score,
        'priority': priority
    }
    
    return PriorityAssignment(priority, container_id, slot_id, score_components)

def is_feasible_assignment(problem: HeuristicProblem, container_id: int, slot_id: int,
                         current_assignments: Dict[int, int]) -> bool:
    """
    Check if a container-slot assignment is feasible
    
    Args:
        problem: Complete problem instance
        container_id: Container ID
        slot_id: Slot ID
        current_assignments: Current assignments
        
    Returns:
        True if assignment is feasible
    """
    container = problem.containers[container_id]
    slot = problem.slots[slot_id]
    
    # Check if slot is already assigned
    if slot_id in current_assignments.values():
        return False
    
    # Check weight capacity
    if container.weight > slot.weight_capacity:
        return False
    
    # Check height limit
    if container.height > slot.height_limit:
        return False
    
    # Check bay weight limits
    bay_weight = 0
    for assigned_container_id, assigned_slot_id in current_assignments.items():
        assigned_slot = problem.slots[assigned_slot_id]
        if assigned_slot.bay_id == slot.bay_id:
            bay_weight += problem.containers[assigned_container_id].weight
    
    bay_weight += container.weight
    if bay_weight > problem.bay_weight_limits[slot.bay_id]:
        return False
    
    return True

print("✅ Priority calculation and feasibility functions defined")

In [ ]:
def priority_based_oog_assignment(problem: HeuristicProblem) -> Tuple[Dict[int, int], List[PriorityAssignment]]:
    """
    Solve OOG container assignment using priority-based heuristic
    
    Args:
        problem: Complete problem instance
        
    Returns:
        Tuple of (assignments, assignment_history)
    """
    assignments = {}  # container_id -> slot_id
    assignment_history = []  # List of assignments in order
    
    # Priority queue for feasible assignments
    priority_queue = []
    
    # Initialize priority queue with all feasible assignments
    for container_id in range(len(problem.containers)):
        for slot_id in range(len(problem.slots)):
            if is_feasible_assignment(problem, container_id, slot_id, assignments):
                priority_assignment = calculate_priority_score(problem, container_id, slot_id, assignments)
                heapq.heappush(priority_queue, priority_assignment)
    
    # Greedy assignment process
    while priority_queue and len(assignments) < len(problem.containers):
        # Get highest priority assignment
        current_assignment = heapq.heappop(priority_queue)
        
        # Check if assignment is still feasible (constraints may have changed)
        if (current_assignment.container_id not in assignments and
            current_assignment.slot_id not in assignments.values() and
            is_feasible_assignment(problem, current_assignment.container_id, 
                                 current_assignment.slot_id, assignments)):
            
            # Make the assignment
            assignments[current_assignment.container_id] = current_assignment.slot_id
            assignment_history.append(current_assignment)
            
            # Update priority queue with new feasible assignments
            # (This is done by checking remaining unassigned containers)
            updated_queue = []
            while priority_queue:
                assignment = heapq.heappop(priority_queue)
                if (assignment.container_id not in assignments and
                    assignment.slot_id not in assignments.values() and
                    is_feasible_assignment(problem, assignment.container_id, 
                                         assignment.slot_id, assignments)):
                    # Recalculate priority with updated assignments
                    updated_assignment = calculate_priority_score(problem, assignment.container_id, 
                                                                assignment.slot_id, assignments)
                    heapq.heappush(updated_queue, updated_assignment)
            
            priority_queue = updated_queue
    
    return assignments, assignment_history

def analyze_heuristic_solution(problem: HeuristicProblem, assignments: Dict[int, int]) -> Dict[str, float]:
    """
    Analyze the quality of the heuristic solution
    
    Args:
        problem: Complete problem instance
        assignments: Final assignments
        
    Returns:
        Dictionary of performance metrics
    """
    metrics = {}
    
    # Total cost
    total_cost = 0
    total_stability = 0
    total_capacity_used = 0
    total_capacity_available = 0
    total_lashing_score = 0
    
    bay_weights = {}
    for bay_id in problem.bay_weight_limits:
        bay_weights[bay_id] = 0
    
    for container_id, slot_id in assignments.items():
        container = problem.containers[container_id]
        slot = problem.slots[slot_id]
        
        total_cost += problem.cost_matrix[container_id, slot_id]
        total_stability += problem.stability_matrix[container_id, slot_id]
        total_capacity_used += container.weight
        total_capacity_available += slot.weight_capacity
        total_lashing_score += container.lashing_coeff * slot.crane_distance
        bay_weights[slot.bay_id] += container.weight
    
    metrics['total_cost'] = total_cost
    metrics['total_stability'] = total_stability
    metrics['capacity_utilization'] = (total_capacity_used / total_capacity_available) * 100
    metrics['lashing_complexity'] = total_lashing_score
    metrics['bay_weights'] = bay_weights
    metrics['assignments_count'] = len(assignments)
    
    return metrics

print("✅ Priority-based heuristic algorithm defined")

In [ ]:
# Solve the OOG assignment problem using priority-based heuristic
assignments, assignment_history = priority_based_oog_assignment(problem)
metrics = analyze_heuristic_solution(problem, assignments)

print("🚀 PRIORITY-BASED OOG ASSIGNMENT ALGORITHM")
print("=" * 50)
print(f"📊 Total assignments made: {len(assignments)} out of {len(containers)} containers")
print(f"💰 Total assignment cost: {metrics['total_cost']} units")
print(f"⚖️  Total stability score: {metrics['total_stability']:.2f}")
print(f"📦 Capacity utilization: {metrics['capacity_utilization']:.1f}%")
print(f"🔗 Lashing complexity: {metrics['lashing_complexity']:.1f}")
print()

# Display assignment sequence with priorities
print("📋 ASSIGNMENT SEQUENCE (in order of selection):")
for i, assignment in enumerate(assignment_history, 1):
    container = problem.containers[assignment.container_id]
    slot = problem.slots[assignment.slot_id]
    print(f"  Step {i}: Container {container.id} → Slot {slot.id}")
    print(f"    Priority: {assignment.priority:.3f}")
    print(f"    Container: {container.container_type}, {container.weight}t, {container.height}m")
    print(f"    Slot: Bay {slot.bay_id}, capacity {slot.weight_capacity}t, height {slot.height_limit}m")
    print(f"    Score components:")
    for component, score in assignment.score_components.items():
        if component != 'priority':
            print(f"      - {component.replace('_', ' ').title()}: {score:.3f}")
    print(f"    Cost: {problem.cost_matrix[assignment.container_id, assignment.slot_id]}, Stability: {problem.stability_matrix[assignment.container_id, assignment.slot_id]:.2f}")
    print()

# Display final bay weight distribution
print("⚓ Bay Weight Distribution:")
for bay_id, weight in metrics['bay_weights'].items():
    limit = problem.bay_weight_limits[bay_id]
    utilization = (weight / limit) * 100
    print(f"  Bay {bay_id}: {weight}t / {limit}t ({utilization:.1f}% utilization)")

In [ ]:
# Compare with mathematical optimization (Tier 1) results
from scipy.optimize import linear_sum_assignment

# Solve using Hungarian algorithm for comparison
row_ind, col_ind = linear_sum_assignment(problem.cost_matrix)
optimal_assignments = {i: j for i, j in zip(row_ind, col_ind)}
optimal_metrics = analyze_heuristic_solution(problem, optimal_assignments)

print("📊 COMPARISON: HEURISTIC vs MATHEMATICAL OPTIMIZATION")
print("=" * 55)
print()
print("🎯 MATHEMATICAL OPTIMIZATION (Hungarian Algorithm):")
print(f"   Total Cost: {optimal_metrics['total_cost']} units")
print(f"   Total Stability: {optimal_metrics['total_stability']:.2f}")
print(f"   Capacity Utilization: {optimal_metrics['capacity_utilization']:.1f}%")
print()
print("🚀 PRIORITY-BASED HEURISTIC:")
print(f"   Total Cost: {metrics['total_cost']} units")
print(f"   Total Stability: {metrics['total_stability']:.2f}")
print(f"   Capacity Utilization: {metrics['capacity_utilization']:.1f}%")
print()

# Calculate performance gaps
cost_gap = ((metrics['total_cost'] / optimal_metrics['total_cost']) - 1) * 100
stability_gap = ((metrics['total_stability'] / optimal_metrics['total_stability']) - 1) * 100
utilization_gap = metrics['capacity_utilization'] - optimal_metrics['capacity_utilization']

print("📈 PERFORMANCE ANALYSIS:")
print(f"   Cost Gap: {cost_gap:+.1f}% ({'Higher' if cost_gap > 0 else 'Lower'} than optimal)")
print(f"   Stability Gap: {stability_gap:+.1f}% ({'Higher' if stability_gap > 0 else 'Lower'} than optimal)")
print(f"   Utilization Gap: {utilization_gap:+.1f}% ({'Higher' if utilization_gap > 0 else 'Lower'} than optimal)")
print()

# Show assignment differences
print("🔄 ASSIGNMENT COMPARISON:")
print("   Optimal vs Heuristic:")
for container_id in range(len(containers)):
    optimal_slot = optimal_assignments[container_id]
    heuristic_slot = assignments.get(container_id, None)
    
    if heuristic_slot is not None:
        status = "✅" if optimal_slot == heuristic_slot else "🔄"
        print(f"   {status} Container {container_id}: Slot {optimal_slot} → Slot {heuristic_slot}")
        if optimal_slot != heuristic_slot:
            opt_cost = problem.cost_matrix[container_id, optimal_slot]
            heu_cost = problem.cost_matrix[container_id, heuristic_slot]
            print(f"      Cost difference: {heu_cost - opt_cost:+d} units")
    else:
        print(f"   ❌ Container {container_id}: Slot {optimal_slot} → Unassigned")

print(f"\n🎯 Heuristic Quality Assessment:")
if cost_gap <= 10:
    print(f"   ✅ Excellent: Cost within 10% of optimal")
elif cost_gap <= 20:
    print(f"   ⚠️  Good: Cost within 20% of optimal")
else:
    print(f"   ❌ Poor: Cost more than 20% above optimal")

In [ ]:
# Create comprehensive visualization of heuristic performance
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Priority-Based OOG Assignment Analysis', fontsize=16, fontweight='bold')

# 1. Priority score breakdown by component
priority_data = []
labels = []
for i, assignment in enumerate(assignment_history):
    components = assignment.score_components
    priority_data.append([components['capacity_utilization'], 
                          components['cost_efficiency'],
                          components['stability_contribution'],
                          components['lashing_simplicity']])
    labels.append(f"C{assignment.container_id}→S{assignment.slot_id}")

priority_array = np.array(priority_data).T
x = np.arange(len(labels))
width = 0.2

components_names = ['Capacity', 'Cost', 'Stability', 'Lashing']
colors = ['skyblue', 'lightgreen', 'lightcoral', 'gold']

for i, (component_name, color) in enumerate(zip(components_names, colors)):
    ax1.bar(x + i * width, priority_array[i], width, label=component_name, alpha=0.8, color=color)

ax1.set_xlabel('Assignments (in order)')
ax1.set_ylabel('Score Component')
ax1.set_title('Priority Score Components by Assignment')
ax1.set_xticks(x + width * 1.5)
ax1.set_xticklabels(labels, rotation=45)
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Heuristic vs Optimal comparison
comparison_metrics = ['Total Cost', 'Total Stability', 'Capacity Utilization']
optimal_values = [optimal_metrics['total_cost'], optimal_metrics['total_stability'], 
                  optimal_metrics['capacity_utilization']]
heuristic_values = [metrics['total_cost'], metrics['total_stability'], 
                    metrics['capacity_utilization']]

x = np.arange(len(comparison_metrics))
width = 0.35

ax2.bar(x - width/2, optimal_values, width, label='Optimal (Hungarian)', alpha=0.8, color='blue')
ax2.bar(x + width/2, heuristic_values, width, label='Heuristic (Priority)', alpha=0.8, color='orange')

ax2.set_xlabel('Performance Metrics')
ax2.set_ylabel('Value')
ax2.set_title('Heuristic vs Optimal Performance')
ax2.set_xticks(x)
ax2.set_xticklabels(comparison_metrics)
ax2.legend()
ax2.grid(True, alpha=0.3)

# Add value labels on bars
for i, (opt, heu) in enumerate(zip(optimal_values, heuristic_values)):
    ax2.text(i - width/2, opt + max(optimal_values) * 0.01, f'{opt:.1f}', ha='center', va='bottom')
    ax2.text(i + width/2, heu + max(heuristic_values) * 0.01, f'{heu:.1f}', ha='center', va='bottom')

# 3. Assignment timeline visualization
timeline_data = []
for i, assignment in enumerate(assignment_history):
    timeline_data.append({
        'step': i + 1,
        'priority': assignment.priority,
        'container_id': assignment.container_id,
        'slot_id': assignment.slot_id,
        'cost': problem.cost_matrix[assignment.container_id, assignment.slot_id]
    })

steps = [d['step'] for d in timeline_data]
priorities = [d['priority'] for d in timeline_data]
costs = [d['cost'] for d in timeline_data]

ax3_twin = ax3.twinx()
ax3.plot(steps, priorities, 'o-', color='purple', linewidth=2, markersize=8, label='Priority Score')
ax3_twin.bar(steps, costs, alpha=0.3, color='red', label='Assignment Cost')

ax3.set_xlabel('Assignment Step')
ax3.set_ylabel('Priority Score', color='purple')
ax3_twin.set_ylabel('Assignment Cost', color='red')
ax3.set_title('Assignment Timeline: Priority Evolution')
ax3.tick_params(axis='y', labelcolor='purple')
ax3_twin.tick_params(axis='y', labelcolor='red')
ax3.grid(True, alpha=0.3)

# Add container labels
for i, d in enumerate(timeline_data):
    ax3.text(d['step'], d['priority'] + 0.02, f'C{d["container_id"]}', 
             ha='center', va='bottom', fontweight='bold')

# 4. Bay weight utilization comparison
bay_ids = list(problem.bay_weight_limits.keys())
optimal_bay_weights = [optimal_metrics['bay_weights'][bid] for bid in bay_ids]
heuristic_bay_weights = [metrics['bay_weights'][bid] for bid in bay_ids]
bay_limits = [problem.bay_weight_limits[bid] for bid in bay_ids]

x = np.arange(len(bay_ids))
width = 0.25

ax4.bar(x - width, bay_limits, width, label='Bay Limit', alpha=0.3, color='gray')
ax4.bar(x, optimal_bay_weights, width, label='Optimal', alpha=0.8, color='blue')
ax4.bar(x + width, heuristic_bay_weights, width, label='Heuristic', alpha=0.8, color='orange')

ax4.set_xlabel('Bay ID')
ax4.set_ylabel('Weight (tons)')
ax4.set_title('Bay Weight Utilization Comparison')
ax4.set_xticks(x)
ax4.set_xticklabels([f'Bay {bid}' for bid in bay_ids])
ax4.legend()
ax4.grid(True, alpha=0.3)

# Add utilization percentages
for i, (opt, heu, limit) in enumerate(zip(optimal_bay_weights, heuristic_bay_weights, bay_limits)):
    opt_util = (opt / limit) * 100
    heu_util = (heu / limit) * 100
    ax4.text(i - width, opt + 1, f'{opt_util:.1f}%', ha='center', va='bottom', fontsize=9)
    ax4.text(i + width, heu + 1, f'{heu_util:.1f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("📊 Comprehensive heuristic performance visualization created")

In [ ]:
# Sensitivity analysis for priority weights
print("🔍 PRIORITY WEIGHTS SENSITIVITY ANALYSIS")
print("=" * 45)

# Test different priority weight configurations
weight_scenarios = [
    {
        'name': 'Balanced (Base)',
        'weights': {'capacity_utilization': 0.30, 'cost_efficiency': 0.25, 
                   'stability_contribution': 0.25, 'lashing_simplicity': 0.20}
    },
    {
        'name': 'Cost-Focused',
        'weights': {'capacity_utilization': 0.20, 'cost_efficiency': 0.40, 
                   'stability_contribution': 0.20, 'lashing_simplicity': 0.20}
    },
    {
        'name': 'Stability-Focused',
        'weights': {'capacity_utilization': 0.25, 'cost_efficiency': 0.20, 
                   'stability_contribution': 0.40, 'lashing_simplicity': 0.15}
    },
    {
        'name': 'Capacity-Focused',
        'weights': {'capacity_utilization': 0.50, 'cost_efficiency': 0.20, 
                   'stability_contribution': 0.20, 'lashing_simplicity': 0.10}
    }
]

results = []

for scenario in weight_scenarios:
    # Create problem with modified weights
    modified_problem = HeuristicProblem(
        problem.containers, problem.slots, problem.cost_matrix, 
        problem.stability_matrix, problem.bay_weight_limits, scenario['weights']
    )
    
    # Solve with modified weights
    scenario_assignments, _ = priority_based_oog_assignment(modified_problem)
    scenario_metrics = analyze_heuristic_solution(modified_problem, scenario_assignments)
    
    results.append({
        'scenario': scenario['name'],
        'cost': scenario_metrics['total_cost'],
        'stability': scenario_metrics['total_stability'],
        'utilization': scenario_metrics['capacity_utilization'],
        'assignments': scenario_metrics['assignments_count']
    })
    
    print(f"\n📊 {scenario['name']}:")
    print(f"   Weights: {scenario['weights']}")
    print(f"   Cost: {scenario_metrics['total_cost']} units")
    print(f"   Stability: {scenario_metrics['total_stability']:.2f}")
    print(f"   Utilization: {scenario_metrics['capacity_utilization']:.1f}%")
    print(f"   Assignments: {scenario_metrics['assignments_count']}/{len(containers)}")

# Create comparison visualization
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Priority Weights Sensitivity Analysis', fontsize=16, fontweight='bold')

scenarios = [r['scenario'] for r in results]
costs = [r['cost'] for r in results]
stabilities = [r['stability'] for r in results]
utilizations = [r['utilization'] for r in results]
assignments = [r['assignments'] for r in results]

# 1. Cost comparison
ax1.bar(scenarios, costs, alpha=0.8, color='lightblue')
ax1.set_ylabel('Total Cost (units)')
ax1.set_title('Cost Performance by Priority Strategy')
ax1.tick_params(axis='x', rotation=45)
for i, cost in enumerate(costs):
    ax1.text(i, cost + max(costs) * 0.01, f'{cost}', ha='center', va='bottom')

# 2. Stability comparison
ax2.bar(scenarios, stabilities, alpha=0.8, color='lightgreen')
ax2.set_ylabel('Total Stability Score')
ax2.set_title('Stability Performance by Priority Strategy')
ax2.tick_params(axis='x', rotation=45)
for i, stability in enumerate(stabilities):
    ax2.text(i, stability + max(stabilities) * 0.01, f'{stability:.2f}', ha='center', va='bottom')

# 3. Utilization comparison
ax3.bar(scenarios, utilizations, alpha=0.8, color='lightcoral')
ax3.set_ylabel('Capacity Utilization (%)')
ax3.set_title('Utilization Performance by Priority Strategy')
ax3.tick_params(axis='x', rotation=45)
for i, util in enumerate(utilizations):
    ax3.text(i, util + max(utilizations) * 0.01, f'{util:.1f}%', ha='center', va='bottom')

# 4. Assignment success comparison
ax4.bar(scenarios, assignments, alpha=0.8, color='gold')
ax4.set_ylabel('Successful Assignments')
ax4.set_title('Assignment Success by Priority Strategy')
ax4.tick_params(axis='x', rotation=45)
ax4.set_ylim(0, len(containers) + 0.5)
for i, assign in enumerate(assignments):
    ax4.text(i, assign + 0.1, f'{assign}/{len(containers)}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print(f"\n🎯 Key Insights from Sensitivity Analysis:")
print(f"   • Priority weight configuration significantly impacts solution quality")
print(f"   • Cost-focused strategy achieves lowest cost but may sacrifice stability")
print(f"   • Stability-focused strategy improves vessel stability at higher cost")
print(f"   • Capacity-focused strategy maximizes utilization but may not be optimal")
print(f"   • Balanced approach provides good compromise across all objectives")

### Why this Tier exists vs earlier Tiers

**Tier 2: Priority-Based Heuristic** offers distinct advantages over mathematical optimization:

**Advantages vs Tier 1 (Mathematical Formulation):**
- **Real-Time Performance**: O(n²m log m) complexity vs O(n³) for Hungarian algorithm
- **Multi-Objective Optimization**: Simultaneously considers cost, stability, capacity, and lashing
- **Dynamic Adaptability**: Priority scores update as assignments are made
- **Operational Flexibility**: Can handle changing constraints and priorities
- **Intuitive Decision Logic**: Priority scores are explainable to operators
- **Partial Solutions**: Can provide good solutions even when full assignment is impossible

**Disadvantages:**
- **No Optimality Guarantee**: May not find the globally optimal solution
- **Parameter Sensitivity**: Performance depends on priority weight selection
- **Local Optima**: Greedy approach can get stuck in suboptimal solutions
- **Weight Tuning Required**: Priority weights need calibration for specific operations

### When to use this Tier

**Use Priority-Based Heuristic when:**
- Real-time decision making is required for vessel operations
- Multiple objectives must be balanced simultaneously
- Problem parameters change dynamically during operations
- Explainable decision logic is important for operators
- Computational efficiency is critical for large-scale problems
- Partial solutions are acceptable when full assignment is infeasible
- Operational constraints require flexible adaptation

**This Tier bridges the gap** between exact mathematical optimization and practical operational requirements, providing solutions that are both computationally efficient and operationally relevant.